In [1]:
import rasterio
from rasterio.mask import mask
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import mapping
import os

In [2]:
flood_volume_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_CASC_Project/Wetland_Climate_Impacts/Flood_Volume_Estimation"
carbon_storage_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_CASC_Project/Wetland_Climate_Impacts/Carbon_Storage_Estimation"

In [3]:
in_file = r"IL_WS_Step17_2024NLCD_WaterRegime_Filter.shp"

In [4]:
in_path = os.path.normpath(os.path.join(flood_volume_folder, in_file))
in_path

'C:\\Users\\Chels\\OneDrive - University of Illinois - Urbana\\Ch5_CASC_Project\\Wetland_Climate_Impacts\\Flood_Volume_Estimation\\IL_WS_Step17_2024NLCD_WaterRegime_Filter.shp'

In [5]:
out_file = r"IL_WS_Step18_2024NLCD_WaterRegime_SoilBiomass_Cstorage"

In [6]:
out_path = os.path.normpath(os.path.join(carbon_storage_folder, out_file))
out_path

'C:\\Users\\Chels\\OneDrive - University of Illinois - Urbana\\Ch5_CASC_Project\\Wetland_Climate_Impacts\\Carbon_Storage_Estimation\\IL_WS_Step18_2024NLCD_WaterRegime_SoilBiomass_Cstorage'

In [7]:
# carbon storage path
value_raster = r"D:\Databases\carbon\WCMC_Biomass_Plus_gNATSGO_SOC_100cm_t_ha_LambertNAD1983.tif"

In [8]:
# read in NWI shapefile
gdf = gpd.read_file(filename=in_path)

In [9]:
# make list for storing C stock estimates
results = []

In [10]:
with rasterio.open(value_raster) as src:
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    cell_area = (src.res[0] * src.res[1])/ (10**4) # ha

    for idx, row in gdf.iterrows():
        geom      = row.geometry
        poly_area = geom.area
        n_cells   = poly_area / cell_area

        if n_cells < 1:
            # very small — sample centroid
            pt     = geom.representative_point()
            coords = [(pt.x, pt.y)]
            val    = list(src.sample(coords))[0][0]
            values = [val] if val != src.nodata else [0]

        else:
            # normal — exact mask
            masked, _ = mask(
                src,
                [mapping(geom)],
                crop=True,
                all_touched=False,
                nodata=0
            )
            values = masked[masked != src.nodata].flatten()
            values = values[~np.isnan(values)].tolist()

        results.append({
            "id":    idx,
            "StockC_Mg":  np.sum(values) * cell_area if values else  0,
            "method": "centroid" if n_cells < 1 else "mask"
        })    

In [11]:
# make results into dataframe
results_df = pd.DataFrame(results)
gdf = gdf.merge(results_df, left_index=True, right_on="id")

In [12]:
# Write the GeoDataFrame to a shapefile
gdf.to_file(os.path.normpath(os.path.join(out_path + ".shp")))

In [13]:
# write the attribute table to a csv file
gdf = gdf.drop(columns=['geometry'])
gdf.to_csv(os.path.normpath(os.path.join(out_path+ ".csv")), index=False)